# Unsloth Core — Sanitization + DeepEval Quality Gate

Clean AI artifacts from generated datasets and validate quality metrics.

In [ ]:
# @title ⚙️ Configuration

NPC_KEY = "history_guide"  # @param ["history_guide", "chef_assistant", "fitness_coach", "astronomy_guide"]
TECHNIQUE = "ollama"  # @param ["template", "ollama", "openai", "anthropic"]
JUDGE_MODEL = "gpt-4o-mini"  # @param {type:"string"}
SKIP_DEEPEVAL = False  # @param {type:"boolean"}

print(f"NPC Key: {NPC_KEY}")
print(f"Technique: {TECHNIQUE}")
print(f"Judge Model: {JUDGE_MODEL}")
print(f"Skip DeepEval: {SKIP_DEEPEVAL}")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/athargamedev/Unsloth_Core.git"
DRIVE_REPO_DIR = "/content/drive/MyDrive/Unsloth_Core"
FALLBACK_REPO_DIR = "/content/Unsloth_Core"

# Detect if we are running in Google Colab
is_colab = False
try:
    import google.colab  # type: ignore
    is_colab = True
except ImportError:
    pass

is_remote_colab = is_colab and os.path.exists("/content")

if is_remote_colab:
    print("Running in remote Google Colab runtime.")
    repo_dir = DRIVE_REPO_DIR
    use_persistent = True
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if os.path.exists("/content/drive/MyDrive"):
            print("Drive mounted and accessible, using persistent storage:", repo_dir)
        else:
            raise OSError("Google Drive mounted but MyDrive is not accessible.")
    except Exception as e:
        repo_dir = FALLBACK_REPO_DIR
        use_persistent = False
        print(f"Drive mount unavailable ({e}), using ephemeral storage:", repo_dir)

    # Robust clone/pull with fallback
    try:
        if not os.path.exists(repo_dir):
            os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
            subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
        else:
            git_dir = os.path.join(repo_dir, ".git")
            if os.path.exists(git_dir) and os.path.isdir(git_dir):
                orig = os.getcwd()
                os.chdir(repo_dir)
                subprocess.run(["git", "pull"], check=False)
                os.chdir(orig)
            else:
                import shutil
                shutil.rmtree(repo_dir)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    except OSError as e:
        print(f"FileSystem Error during persistent storage access: {e}")
        if use_persistent:
            print("Falling back immediately to ephemeral storage to prevent crash...")
            repo_dir = FALLBACK_REPO_DIR
            if not os.path.exists(repo_dir):
                os.makedirs(os.path.dirname(repo_dir), exist_ok=True)
                subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)

    os.chdir(repo_dir)
else:
    print("Running locally in the editor/workspace.")
    curr = Path(os.getcwd()).resolve()
    repo_dir = None
    for parent in [curr] + list(curr.parents):
        if (parent / "ucore").exists() and (parent / "scripts").exists():
            repo_dir = parent
            break
    if repo_dir:
        print("Detected local repository root at:", repo_dir)
        os.chdir(repo_dir)
    else:
        print("Error: Could not find local repository root containing ucore!")
        sys.exit(1)

print("Current working directory:", os.getcwd())

# ── Install dependencies ──
print("Installing DeepEval...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepeval"], check=True)

# ── Configure judge credentials ──
judge_is_openai = JUDGE_MODEL.startswith("gpt-") or JUDGE_MODEL.startswith("o")

if judge_is_openai:
    if is_remote_colab:
        try:
            from google.colab import userdata  # type: ignore
            api_key = userdata.get("OPENAI_API_KEY")
            if api_key:
                os.environ["OPENAI_API_KEY"] = api_key
                print("Loaded OPENAI_API_KEY from Colab secrets.")
        except Exception:
            pass

    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
    print("OpenAI judge model configured.")
else:
    print("WARNING: Using Ollama judge model inside Colab.")
    print("Colab runtimes do NOT have Ollama installed by default.")
    print("Either:")
    print("  1. Use an OpenAI model (gpt-4o-mini, etc.) as JUDGE_MODEL.")
    print("  2. Install and start Ollama in Colab (advanced, not recommended).")

In [ ]:
import sys
import subprocess
from pathlib import Path

NPC_KEY = globals().get("NPC_KEY", "history_guide")
TECHNIQUE = globals().get("TECHNIQUE", "template")

train_path = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}/train.jsonl"
clean_path = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}/train_clean.jsonl"

print(f"Raw dataset:  {train_path}")
print(f"Clean output: {clean_path}")
print()

if not Path(train_path).exists():
    print(f"ERROR: Raw dataset not found at {train_path}")
    print(f"Generate it first with: ./ucore generate subjects/{NPC_KEY}.json --technique {TECHNIQUE}")
    sys.exit(1)

# Count before rows
before_count = 0
with open(train_path) as f:
    for line in f:
        line = line.strip()
        if line:
            before_count += 1
print(f"Rows before sanitization: {before_count}")
print()

# Run sanitize
python_bin = sys.executable
sanitize_cmd = [
    python_bin, "scripts/sanitize_dataset.py",
    train_path,
    "--output", clean_path,
    "--strict-canonical",
]
print(f"Running: {' '.join(sanitize_cmd)}")
print()
result = subprocess.run(sanitize_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("=== SANITIZATION FAILED ===")
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
    sys.exit(1)
else:
    print(result.stdout)

# Count after rows
after_count = 0
if Path(clean_path).exists():
    with open(clean_path) as f:
        for line in f:
            line = line.strip()
            if line:
                after_count += 1

print(f"Rows before sanitization: {before_count}")
print(f"Rows after sanitization:  {after_count}")
print(f"Rows removed:             {before_count - after_count}")
print(f"Retention rate:           {after_count / before_count:.1%}" if before_count else "N/A")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

if SKIP_DEEPEVAL:
    print("SKIP_DEEPEVAL is True -- skipping DeepEval quality gate.")
else:
    NPC_KEY = globals().get("NPC_KEY", "history_guide")
    TECHNIQUE = globals().get("TECHNIQUE", "template")
    JUDGE_MODEL = globals().get("JUDGE_MODEL", "gpt-4o-mini")

    judge_is_openai = JUDGE_MODEL.startswith("gpt-") or JUDGE_MODEL.startswith("o")

    spec_path = f"subjects/{NPC_KEY}.json"
    if not Path(spec_path).exists():
        print(f"ERROR: Spec not found at {spec_path}")
        sys.exit(1)

    quality_summary_path = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}/quality_summary.json"

    # Build env vars
    env = os.environ.copy()
    env["DEEPEVAL_DATASET_NPC_KEYS"] = NPC_KEY
    env["DEEPEVAL_DATASET_TECHNIQUE"] = TECHNIQUE
    env["DEEPEVAL_DATASET_CASES_PER_CATEGORY"] = "2"
    env["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"

    if judge_is_openai:
        env["DEEPEVAL_OPENAI_MODEL"] = JUDGE_MODEL
    else:
        env["DEEPEVAL_OLLAMA_MODEL"] = JUDGE_MODEL

    python_bin = sys.executable
    eval_cmd = [
        python_bin, "scripts/dataset_eval.py",
        spec_path,
        "--technique", TECHNIQUE,
        "--judge-model", JUDGE_MODEL,
        "--cases-per-category", "2",
        "--soft-fail",
        "--output", quality_summary_path,
    ]

    print(f"Running: {' '.join(eval_cmd)}")
    print()
    result = subprocess.run(eval_cmd, env=env, capture_output=True, text=True)

    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        print(f"DeepEval exited with code {result.returncode}")
        sys.exit(result.returncode)

    print()
    print("--- DeepEval Summary ---")
    print(f"Judge model: {JUDGE_MODEL}")
    print(f"Output:      {quality_summary_path}")

In [ ]:
import json
from pathlib import Path

NPC_KEY = globals().get("NPC_KEY", "history_guide")
TECHNIQUE = globals().get("TECHNIQUE", "template")

quality_dir = Path(f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}")
summary_path = quality_dir / "quality_summary.json"
failures_path = quality_dir / "quality_failures.json"

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("=== Quality Summary ===")
    print(f"Passed:    {summary.get('passed', 'N/A')} / {summary.get('total', 'N/A')}")
    print(f"Pass rate: {summary.get('pass_rate', 'N/A')}")
    print(f"Judge:     {summary.get('judge_model', 'N/A')}")
    print()

    metrics = summary.get("metrics", {})
    if metrics:
        print("--- Per-Metric Scores ---")
        any_below = False
        for name, data in sorted(metrics.items()):
            avg_score = data.get("avg_score", 0)
            threshold = data.get("threshold", 0)
            passed = data.get("passed", 0)
            total = data.get("count", 0)
            status = "PASS" if avg_score >= threshold else "FAIL"
            if avg_score < threshold:
                any_below = True
            print(f"  {name:30s} avg={avg_score:.3f}  threshold={threshold:.2f}  [{status}]  ({passed}/{total})")

        if any_below:
            print()
            print("WARNING: Some metrics are below threshold! Consider regenerating the dataset.")

    categories = summary.get("categories", {})
    if categories:
        print()
        print("--- Per-Category Results ---")
        for cat, data in sorted(categories.items()):
            print(f"  {cat:25s}  {data.get('passed', 0)}/{data.get('total', 0)} passed")
else:
    print(f"No quality summary found at {summary_path}")

print()

if failures_path.exists():
    failures = json.loads(failures_path.read_text())
    print(f"=== Quality Failures ({len(failures)} entries) ===")
    for i, fcase in enumerate(failures[:10], 1):
        category = fcase.get("metadata", {}).get("category", "?") if isinstance(fcase.get("metadata"), dict) else "?"
        concept = fcase.get("metadata", {}).get("concept", "?") if isinstance(fcase.get("metadata"), dict) else "?"
        failures_metrics = fcase.get("metricsData", fcase.get("metrics_data", []))
        if failures_metrics:
            names = ", ".join(m.get("name", "?") for m in failures_metrics if isinstance(m, dict))
        else:
            names = "?"
        print(f"  {i}. category={category}  concept={concept}  metrics=[{names}]")
    if len(failures) > 10:
        print(f"  ... and {len(failures) - 10} more failures")

In [ ]:
from google.colab import files
import os

NPC_KEY = globals().get("NPC_KEY", "history_guide")
TECHNIQUE = globals().get("TECHNIQUE", "template")
SKIP_DEEPEVAL = globals().get("SKIP_DEEPEVAL", False)

ds_dir = f"subjects/datasets/{NPC_KEY}/{TECHNIQUE}"

# Always download the clean dataset
clean_path = os.path.join(ds_dir, "train_clean.jsonl")
if os.path.exists(clean_path):
    print(f"Downloading: {clean_path}")
    files.download(clean_path)
else:
    print(f"Not found: {clean_path}")

if not SKIP_DEEPEVAL:
    for fname in ["quality_summary.json", "quality_failures.json"]:
        fpath = os.path.join(ds_dir, fname)
        if os.path.exists(fpath):
            print(f"Downloading: {fpath}")
            files.download(fpath)
        else:
            print(f"Not found: {fpath}")

print()
print("Download complete.")